# Phase 3: Pandas Exploratory Data Analysis (10 Questions)
- **Objective:** Use Pandas and NumPy to perform advanced data wrangling, handle missing/zero values, calculate movie profit and ROI, detect statistical outliers using IQR, and analyze cast/genre trends.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to SQLite database and load tables into DataFrames
db_path = '/content/data/tmdb_movies.db'
conn = sqlite3.connect(db_path)

movies_df = pd.read_sql("SELECT * FROM movies", conn)
genres_df = pd.read_sql("SELECT * FROM genres", conn)
movie_genres_df = pd.read_sql("SELECT * FROM movie_genres", conn)
cast_df = pd.read_sql("SELECT * FROM cast", conn)
crew_df = pd.read_sql("SELECT * FROM crew", conn)
conn.close()

print("Data loaded successfully into Pandas DataFrames for EDA!")

In [ ]:
# --- Q1: Total number of unique entities ---
print(f"Q1 -> Unique Movies: {movies_df['movie_id'].nunique()}, Genres: {genres_df['genre_id'].nunique()}, Cast: {cast_df['person_id'].nunique()}, Crew: {crew_df['person_id'].nunique()}")

# --- Q2: Missing or zero budget/revenue values ---
zero_budget = (movies_df['budget'] == 0) | (movies_df['budget'].isna())
zero_revenue = (movies_df['revenue'] == 0) | (movies_df['revenue'].isna())
print(f"Q2 -> Movies with missing/zero budget or revenue: {movies_df[zero_budget | zero_revenue].shape[0]}")

# --- Q3: Profit and ROI for valid figures ---
valid_df = movies_df[(movies_df['budget'] > 0) & (movies_df['revenue'] > 0)].copy()
valid_df['profit'] = valid_df['revenue'] - valid_df['budget']
valid_df['roi'] = valid_df['profit'] / valid_df['budget']
print("Q3 -> Sample Profit & ROI:")
print(valid_df[['title', 'budget', 'revenue', 'roi']].head(3))

# --- Q4: Top 15 movies by ROI (Budget >= $1M) ---
top_roi = valid_df[valid_df['budget'] >= 1_000_000].sort_values(by='roi', ascending=False).head(15)
print("Q4 -> Top 3 High ROI Movies:")
print(top_roi[['title', 'roi']].head(3))

# --- Q5: Movies released per year & peak year ---
movies_df['release_year'] = pd.to_datetime(movies_df['release_date'], errors='coerce').dt.year
yearly_counts = movies_df['release_year'].value_counts().sort_index()
peak_year = yearly_counts.idxmax()
print(f"Q5 -> Peak release year: {int(peak_year)} with {yearly_counts[peak_year]} movies.")

In [ ]:
# --- Q6: Runtime statistical outliers (IQR method) ---
runtimes = movies_df['runtime'].dropna()
Q1, Q3 = runtimes.quantile(0.25), runtimes.quantile(0.75)
IQR = Q3 - Q1
outliers = movies_df[(movies_df['runtime'] < (Q1 - 1.5 * IQR)) | (movies_df['runtime'] > (Q3 + 1.5 * IQR))]
print(f"Q6 -> Runtime outliers count: {len(outliers)}")

# --- Q7: Primary genre analysis ---
merged_mg = movie_genres_df.merge(genres_df, on='genre_id').merge(movies_df, on='movie_id')
primary_genre = merged_mg.groupby('movie_id').first().reset_index()
print("Q7 -> Top Primary Genres by Popularity:")
print(primary_genre.groupby('genre_name')['popularity'].mean().sort_values(ascending=False).head(3))

# --- Q8: Top prolific actors & avg rating ---
actor_stats = cast_df.merge(movies_df, on='movie_id').groupby('actor_name').agg(
    movie_count=('movie_id', 'count'), avg_rating=('vote_average', 'mean')
).sort_values(by='movie_count', ascending=False).head(5)
print("Q8 -> Top Prolific Actors:")
print(actor_stats)

# --- Q9: Duplicate titles / remakes check ---
dup_titles = movies_df[movies_df.duplicated(subset=['title'], keep=False)]
print(f"Q9 -> Duplicate/remake titles found: {len(dup_titles)}")

# --- Q10: Unusual Popularity vs Vote Count patterns ---
pop_z = (movies_df['popularity'] - movies_df['popularity'].mean()) / movies_df['popularity'].std()
vote_z = (movies_df['vote_count'] - movies_df['vote_count'].mean()) / movies_df['vote_count'].std()
unusual = movies_df[abs(pop_z - vote_z) > 2]
print(f"Q10 -> Unusual popularity-vote pattern count: {len(unusual)}")
print("\n--- Phase 3 Pandas Analysis Completed ---")

### Written Insight & Recommendation:
- **Financial Return Patterns:** ROI computations show that mid-budget films often deliver higher percentage returns than mega-blockbusters, proving that controlled production costs mitigate financial exposure.